In [4]:
import os 
import matplotlib.pyplot as plt
import mne
import numpy as np
from pynwb import NWBHDF5IO

directory = '/Users/nicole.burke/Library/CloudStorage/OneDrive-ChildMindInstitute/02_Projects/06_rockland_sample/01_rockland_descriptor_paper/download_data'
os.chdir(directory)
print(os.getcwd())

/Users/nicole.burke/Library/CloudStorage/OneDrive-ChildMindInstitute/02_Projects/06_rockland_sample/01_rockland_descriptor_paper/download_data


## Extract data from .nwb file and get 'mne' ready

Check which datastructure 

In [7]:
nwb_file_path = "sub-A00014880/sub-A00014880_ses-MOBI1A_task-passivesherlock_run-01_MoBI.nwb" 

with NWBHDF5IO(nwb_file_path, 'r') as io:
    nwbfile = io.read()
    print("SUBJECT:", nwb_file_path)
    print(list(nwbfile.acquisition.keys()))

SUBJECT: sub-A00014880/sub-A00014880_ses-MOBI1A_task-passivesherlock_run-01_MoBI.nwb
['ElectricalSeries', 'Left_eye_gaze_1', 'Pupil_Diameters_EL_1', 'Pupil_Rotation_EL_1', 'Right_eye_gaze_1', 'StimLabels', 'allOpenSignalsData_2']


In [ ]:
with NWBHDF5IO(nwb_file_path, "r") as io:
    nwbfile = io.read()
    # Get container 
    obj = nwbfile.acquisition["Right_eye_gaze_1"]
    # Get the SpatialSeries
    cst = obj.spatial_series['Right_eye_gaze'].data[:]
    # Timestamps for each data point
    times = obj.spatial_series['Right_eye_gaze'].timestamps[:]
    # Description for cst data holds all the headers
    headers =obj.spatial_series['Right_eye_gaze'].description.split(',')
    print(headers)
    # Hard code sampling frequency 
    sfreq = 500

# MNE expects data shaped as (n_channels, n_times)
# Transpose the matrix so it becomes 2 rows (X and Y) by N time samples
gaze_data_mne = cst.T

# Define channels for MNE
ch_names = ["rightEyeX", "rightEyeY"]
ch_types = ["eyegaze", "eyegaze"]

# Create MNE Info and RawArray structures
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
raw_et = mne.io.RawArray(gaze_data_mne, info)

# Handle any absolute NaN values in raw data by replacing them with 0 (MNE needs actual numbers)
raw_et._data = np.nan_to_num(raw_et._data, nan=0.0)

print("Raw ET:")
print(raw_et)
print("*"*40)
print("Info:")
print(info)

['rightEyeX', 'rightEyeY']
Creating RawArray with float64 data, n_channels=2, n_times=308687
    Range : 0 ... 308686 =      0.000 ...   617.372 secs
Ready.
Raw ET:
<RawArray | 2 x 308687 (617.4 s), ~4.7 MiB, data loaded>
****************************************
Info:
<Info | 7 non-empty values
 bads: []
 ch_names: rightEyeX, rightEyeY
 chs: 2 Eye-tracking (Gaze position)
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 250.0 Hz
 meas_date: unspecified
 nchan: 2
 projs: []
 sfreq: 500.0 Hz
>


## Apply low pass filter 

In [ ]:
# From documentation: 
    # l_freq < h_freq: band-pass filter
    # l_freq > h_freq: band-stop filter
    # l_freq is not None and h_freq is None: high-pass filter
    # l_freq is None and h_freq is not None: low-pass filter
ch_indices = range(len(raw_et.ch_names))
print(ch_indices)

raw_et_filtered = raw_et.copy().filter(
    l_freq = None, h_freq=30, picks=ch_indices, method="fir", phase="zero"
)

Applying low-pass filter...
range(0, 2)
No data channels found. The highpass and lowpass values in the measurement info will not be updated.
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 221 samples (0.442 s)

